# Perturb-Seq

Perturb-seq is a high-throughput method that combines CRISPR-based genetic perturbations with single-cell RNA sequencing to measure how specific gene disruptions affect cellular transcriptional programs. By linking perturbations to transcriptomic readouts at the single-cell level, it enables systematic mapping of gene function, pathways, and regulatory networks in complex biological systems.

In this notebook, you will practice working with Perturb-Seq data in KRAS. 

In [ ]:
!pip install pandas anndata scipy scanpy numpy seaborn matplotlib

In [ ]:
# Import necessary packages

import pandas as pd
import anndata as ad
from scipy.io import mmread
import scipy.sparse
import scanpy as sc
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ranksums
from scipy import stats
import os
import zipfile

In [ ]:
from pathlib import Path

wd = str(Path.cwd().resolve())

In [ ]:
#Unzip your data file

zip_file_path = 'data.zip' # Your actual zip file name/path
extract_dir = 'data' # Name of the folder to extract to

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

## Loading the processed matrix

AnnData is a data structure designed for single-cell omics, storing a large expression matrix (cells × genes) along with associated metadata. The rows (.obs) represent cells with annotations like cell type or QC metrics, while the columns (.var) represent genes with their identifiers and features. It also supports layers, embeddings, and uns, making it a flexible container for analysis workflows in Scanpy and related tools.

In [ ]:
# The .mtx file stores the raw counts of how many transcripts of each gene were detected in each cell
X = mmread(f"{wd}/data/GSE161824_A549_KRAS.processed.matrix.mtx").tocsr()

# Load gene names and barcodes (metadata for the matrix)
genes = pd.read_csv(f"{wd}/data/GSE161824_A549_KRAS.processed.genes.csv", header=None)
barcodes = pd.read_csv(f"{wd}/data/GSE161824_A549_KRAS.processed.cells.csv", header=None)

In [ ]:
# Create an AnnData object to hold the single-cell expression matrix
adata = ad.AnnData(X)

# Assign gene names to the variable axis (columns of the expression matrix)
adata.var_names = genes[0].values

# Assign cell barcodes to the observation axis (rows of the expression matrix)
adata.obs_names = barcodes[0].values

In [ ]:
# Calculate basic quality control (QC) metrics for each cell and gene
sc.pp.calculate_qc_metrics(adata, inplace=True)

# Filter out low-quality cells:
adata = adata[adata.obs['n_genes_by_counts'] > 200, :]

# Remove cells with very high transcript counts (>25,000), which may indicate doublets
adata = adata[adata.obs['total_counts'] < 25000, :]

# Filter out lowly expressed genes:
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
# Look at the structure of the object
adata

Here’s what that output means, broken down:

AnnData object with n_obs × n_vars = 90262 × 1145
This dataset has 90,262 observations (cells) and 1,145 variables (genes) in its expression matrix. Rows = cells, columns = genes.

obs: (cell-level metadata)
Each entry in .obs is a column of annotations describing the cells. For example:
- n_genes_by_counts: number of genes detected in each cell.
- total_counts: total UMIs (transcript counts) per cell.
- pct_counts_in_top_X_genes: % of counts explained by the top X most highly expressed genes in that cell (QC metric).

var: (gene-level metadata)
Each entry in .var is a column describing the genes. For example:
- n_cells_by_counts: number of cells in which a gene is detected.
- mean_counts: average expression of that gene across cells.
- pct_dropout_by_counts: % of cells where the gene was not detected (dropouts).

## Wild Type vs Variant Data

Let's add an "obs" that tells us whether a given cell is wildtype, or has a variant introduced

In [ ]:
# Import a dataframe that matches variants to cell barcodes
variant2cell = pd.read_csv(f'{wd}/data/cell_variant_info.csv')
variant2cell.head()

Here, if the guide identity is unassigned, that means a variant was not introduced to the cell. So it is wildtype (WT). If the guide identity has a variant, that is the variant that was introduced to the cell.

In [ ]:
# Define a function that returns WT if the value is unassigned, and extracts the residue position using
# regex otherwise
def extract_variant(x):
    if x == "KRAS_unassigned":
        return "WT"
    else:
        # Extract all digits from the string
        num = ''.join(filter(str.isdigit, x))
        return num

variant2cell["variant"] = variant2cell["guide_identity"].apply(extract_variant)

variant2cell

In [ ]:
# Let's make a set that contains the cell barcodes of WT cells
WT_set = set(variant2cell[variant2cell['variant']=='WT']['cell_barcode'])

In [ ]:
# Now we can add the new observation called "mut" to the original anndata object
adata.obs['mut'] = 'Variant'  # default
adata.obs.loc[adata.obs_names.isin(WT_set), 'mut'] = 'WT'

In [ ]:
# We can see that "mut" was added to our obs
adata

## Visualizing the Effects of Variants on Cells

KRAS is a small GTPase that acts as a molecular switch upstream of the MAPK signaling pathway. When KRAS is mutated into an oncogenic form, it becomes constitutively active, driving continuous signaling through RAF, MEK, and ERK kinases. This persistent MAPK pathway activation promotes uncontrolled cell proliferation, survival, and differentiation, contributing to tumorigenesis. In cancers such as pancreatic, lung, and colorectal, KRAS mutations are a major driver of MAPK pathway hyperactivation and therapeutic resistance.

Here we will try to visualize the effects of KRAS variants on the MAPK pathway.

In [ ]:
# Load in the MAPK gene set as a list
mapk = pd.read_csv(f'{wd}/data/MAPK_gene_set.csv')['0'].tolist()
print(f"There are {len(mapk)} genes in the gene set.")

# Check which genes have expression data
gene_list = [g for g in mapk if g in adata.var_names]
print(f"Using {len(gene_list)} genes from your gene set.")

In [ ]:
# Subset the expression matrix to only include the genes in 'gene_list'
X = adata[:, gene_list].X

# Convert to a dense NumPy array if the matrix is sparse
if not isinstance(X, np.ndarray):
    X = X.toarray()

# Compute a gene set enrichment score for each cell
# - gene_list: the set of genes whose combined expression you want to score
# - score_name: name of the new column in adata.obs to store the scores
# The score is calculated as the average expression of the genes in the list minus
# the average expression of a reference set of genes with similar expression levels,
# which helps normalize for library size and other technical differences.
sc.tl.score_genes(adata, gene_list, score_name='gene_set_enrichment_score')

In [ ]:
# Look at the structure of adata now
adata

In [ ]:
# Create a bar plot of average gene set enrichment scores per mutation group
sns.barplot(
    data=adata.obs,                     # use the cell metadata DataFrame from AnnData
    x='mut',                            # plot mutation groups ('WT', 'Variant') on the x-axis
    y='gene_set_enrichment_score',      # plot the GSEA score on the y-axis
    errorbar=('se', 1),                 # show standard error of the mean (SEM) as error bars
    hue='mut',                          # assign hue to match x values to avoid Seaborn warnings
    palette='muted',                    # use a muted color palette for the bars
    dodge=False                          # prevent splitting bars because hue duplicates x
)

# Add axis labels and title for clarity
plt.xlabel('Group')                      # x-axis label
plt.ylabel('Average GSEA Score')         # y-axis label
plt.title('Average GSEA Score by Group with Error Bars')  # plot title

# Display the plot
plt.show()

In [ ]:
# Create a violin plot to visualize the distribution of gene set enrichment scores by group
sns.violinplot(
    data=adata.obs,                    # use the cell metadata from AnnData
    x="mut",                           # mutation groups ('WT', 'Variant') on the x-axis
    y="gene_set_enrichment_score",     # GSEA score on the y-axis
    inner="box"                       # show a small box plot inside each violin (median and quartiles)
)

# Explanation:
# - The violin shape shows the **full distribution** of scores for each group.
# - The inner box provides summary statistics (median, quartiles) within the distribution.
# - Useful for visualizing variability and differences between WT and Variant cells.

## Exercise

KRAS, when activated, not only stimulates the MAPK pathway but also signals through the PI3K–AKT–mTOR pathway, a central regulator of cell growth and survival. Activated KRAS recruits and activates PI3K, which converts PIP2 to PIP3, leading to the recruitment and activation of AKT. AKT then phosphorylates multiple downstream targets, including mTOR, promoting protein synthesis, cell growth, metabolism, and survival. Oncogenic KRAS mutations can therefore hyperactivate PI3K–AKT–mTOR signaling, contributing to uncontrolled proliferation, resistance to apoptosis, and tumorigenesis.

Try repeating the above on your own for the PI3K-AKT-mTOR pathway.

In [ ]:
pi3k = pd.read_csv(f'{wd}/data/PI3K-AKT-mTOR_gene_set.csv')['0'].tolist()

This workflow was developed by Kriti Shukla at the University of North Carolina at Chapel Hill. The data used in this workflow is from Ursu O, Neal JT, Shea E, Thakore PI, Jerby-Arnon L, Nguyen L, Dionne D, Diaz C, Bauman J, Mosaad MM, Fagre C, Lo A, McSharry M, Giacomelli AO, Ly SH, Rozenblatt-Rosen O, Hahn WC, Aguirre AJ, Berger AH, Regev A, Boehm JS. Massively parallel phenotyping of coding variants in cancer with Perturb-seq. Nat Biotechnol. 2022 Jun;40(6):896-905. doi: 10.1038/s41587-021-01160-7. Epub 2022 Jan 20. Erratum in: Nat Biotechnol. 2022 Nov;40(11):1691. doi: 10.1038/s41587-022-01495-9. PMID: 35058622